# Modelagem e Análise Formal da Esteira Seletora com Redes de Petri
### Árvore de Cobertura (Karp-Miller com $\omega$) • Árvore de Atingibilidade Finita • Prevenção de Deadlocks e Transbordo
**Disciplina:** ECAA08 • Engenharia de Controle e Automação & Matemática Discreta  
**Unidade de Processo:** Célula de Manufatura Flexível (FMS) — Dinâmica de Infeed (`XV-101`), Inspeção (`INSP-200`), Triagem Pneumática (`XV-201`), Canaleta (`CX-301`) e Proteção Anti-Transbordo (`FULL-300`)  

---

## 🎯 Objetivos do Notebook
1. **Modelagem Matricial de Redes de Petri:** Construir a quíntupla formal $N = (P, T, Pre, Post, M_0)$ e calcular a Matriz de Incidência $C = Post - Pre$ da esteira seletora de peças.
2. **Demonstração do Crescimento Ilimitado (Árvore de Cobertura de Karp-Miller):** Simular a operação sem controle de capacidade de canaleta, comprovando o surgimento automático do símbolo $\omega$ (ômega) no lugar de acúmulo.
3. **Síntese de Lugar Complementar (Invariante-P):** Adicionar o lugar $p_6$ (`CAP-301`, capacidade $K=3$) e comprovar a conservação linear $M(p_5) + M(p_6) = 3$.
4. **Geração da Árvore de Atingibilidade Finita:** Explorar deterministicamente o espaço de estados e construir o Grafo de Transição de Marcações (*Reachability Graph*).
5. **Auditoria Algébrica com Asserções (`assert`):** Validar ausência de *deadlocks*, $k$-limitação, segurança (*safeness*) e vivacidade do sistema.
6. **Visualização Gráfica do Grafo de Atingibilidade:** Plotar a máquina de estados discretos no NetworkX destacando os estados de operação e de canaleta cheia.

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from collections import deque
from typing import List, Tuple, Dict, Set, Optional, Any, Union

# Configuração estética dos gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 110

print("✅ Módulos importados com sucesso para a análise formal de Redes de Petri na FMS.")

---
## 1. Modelagem Matricial da Rede de Petri da Esteira Seletora

Mapeamos os $6$ lugares e $4$ transições fundamentais do processo:
- Lugares:
  - $p_1$: `SILO_DISP` (Peças disponíveis para alimentação no silo `SILO-100`)
  - $p_2$: `ESTEIRA_LIVRE` (Vaga na esteira principal `M-101`, garante 1 peça por trecho)
  - $p_3$: `PECA_INSPECAO` (Peça no túnel óptico `INSP-200` sendo analisada)
  - $p_4$: `PISTAO_PRONTO` (Pistão desviador `XV-201` recuado e pronto)
  - $p_5$: `PECA_CANALETA` (Peças acumuladas na canaleta de saída `CX-301`)
  - $p_6$: `VAGAS_CANALETA` (Capacidade livre restante na canaleta delimitada por `FULL-300`)
- Transições:
  - $t_1$: `T1_ALIMENTAR` (Pistão `XV-101` empurra peça para a esteira)
  - $t_2$: `T2_DESVIAR` (Pistão `XV-201` avança e desvia peça para a canaleta `CX-301`)
  - $t_3$: `T3_COLETAR` (AGV ou operador recolhe uma peça da canaleta)
  - $t_4$: `T4_REABASTECER` (Chegada de novo lote de tarugos no silo de entrada)

In [ ]:
class RedePetriFMS:
    """Classe formal de Rede de Petri Marcada para a Célula de Manufatura Flexível."""
    def __init__(
        self,
        nomes_lugares: List[str],
        nomes_transicoes: List[str],
        matriz_pre: np.ndarray,
        matriz_post: np.ndarray,
        m0: Tuple[int, ...]
    ):
        self.lugares = nomes_lugares
        self.transicoes = nomes_transicoes
        self.pre = np.array(matriz_pre, dtype=int)
        self.post = np.array(matriz_post, dtype=int)
        self.C = self.post - self.pre  # Matriz de Incidência
        self.m0 = tuple(m0)
        
        assert self.pre.shape == (len(self.lugares), len(self.transicoes)), "Dimensão inválida de Pre"
        assert self.post.shape == (len(self.lugares), len(self.transicoes)), "Dimensão inválida de Post"
        assert len(self.m0) == len(self.lugares), "Dimensão de M0 diverge do número de lugares"

    def habilitada(self, M: Tuple[Union[int, str], ...], t_idx: int) -> bool:
        """Verifica se a transição t_idx está habilitada na marcação M (suporta omega)."""
        for p_idx in range(len(self.lugares)):
            val_p = M[p_idx]
            req = self.pre[p_idx, t_idx]
            if req > 0:
                if val_p == 'omega':
                    continue
                elif val_p < req:
                    return False
        return True

    def disparar(self, M: Tuple[Union[int, str], ...], t_idx: int) -> Tuple[Union[int, str], ...]:
        """Dispara a transição e retorna a nova marcação M' via equação de estado."""
        nova_M = list(M)
        for p_idx in range(len(self.lugares)):
            if nova_M[p_idx] != 'omega':
                nova_M[p_idx] = nova_M[p_idx] + int(self.C[p_idx, t_idx])
        return tuple(nova_M)

print("✅ Classe RedePetriFMS compilada com sucesso.")

---
## 2. Parte A: Demonstração de Sistema Aberto e Árvore de Cobertura com $\omega$

Construímos a rede da esteira **sem o controle de vagas da canaleta** (sistema aberto de 5 lugares). Se a alimentação for mais rápida que a coleta, a quantidade de peças em $p_5$ cresce arbitrariamente. O Algoritmo de Karp-Miller detecta esse crescimento através do símbolo **$\omega$**.

In [ ]:
lugares_aberta = ["p1_SILO", "p2_ESTEIRA", "p3_TUNEL", "p4_PISTAO", "p5_CANALETA"]
transicoes_aberta = ["t1_ALIMENTAR", "t2_DESVIAR", "t3_COLETAR", "t4_REABASTECER"]

# Matriz Pre: [p, t]
pre_aberta = [
    [1, 0, 0, 0],  # p1: consome em t1
    [1, 0, 0, 0],  # p2: consome em t1 (esteira ocupada)
    [0, 1, 0, 0],  # p3: consome em t2
    [0, 1, 0, 0],  # p4: consome em t2 (pistão avança)
    [0, 0, 1, 0],  # p5: consome em t3 (coleta do operador)
]

# Matriz Post: [p, t]
post_aberta = [
    [0, 0, 0, 1],  # p1: reabastecido em t4
    [0, 1, 0, 0],  # p2: liberado em t2
    [1, 0, 0, 0],  # p3: produzido em t1
    [0, 1, 0, 0],  # p4: restituído em t2
    [0, 1, 0, 0],  # p5: peça cai na canaleta em t2 (SEM LIMITE!)
]

m0_aberta = (1, 1, 0, 1, 0)  # 1 peça no silo, esteira livre, pistão pronto, canaleta vazia
rede_aberta = RedePetriFMS(lugares_aberta, transicoes_aberta, pre_aberta, post_aberta, m0_aberta)

df_incidencia_aberta = pd.DataFrame(rede_aberta.C, index=lugares_aberta, columns=transicoes_aberta)
print("=== MATRIZ DE INCIDÊNCIA C (REDE ABERTA - SEM CAPACIDADE) ===")
display(df_incidencia_aberta)

### Execução do Algoritmo de Karp-Miller com Injeção de $\omega$

In [ ]:
def construir_arvore_cobertura(rede: RedePetriFMS, max_nos: int = 50) -> Tuple[List[dict], bool]:
    """
    Algoritmo de Karp-Miller para geração da Árvore de Cobertura com símbolo omega.
    Retorna os nós gerados e um booleano indicando se omega foi introduzido.
    """
    nos_gerados = []
    fila = deque()
    
    raiz = {
        'id': 0,
        'M': rede.m0,
        'parent_id': None,
        'transicao': None,
        'caminho': [rede.m0]
    }
    fila.append(raiz)
    nos_gerados.append(raiz)
    omega_detectado = False
    
    while fila and len(nos_gerados) < max_nos:
        curr = fila.popleft()
        M_curr = curr['M']
        
        for t_idx, t_nome in enumerate(rede.transicoes):
            if rede.habilitada(M_curr, t_idx):
                M_next = rede.disparar(M_curr, t_idx)
                M_next_list = list(M_next)
                
                # Verificação de cobertura de ancestrais (Regra de Karp-Miller)
                for anc_M in curr['caminho']:
                    # Verifica se M_next >= anc_M com estrita superioridade em pelo menos uma coordenada
                    maior_ou_igual = True
                    estritamente_maior = False
                    
                    for p in range(len(rede.lugares)):
                        v_anc = anc_M[p]
                        v_nxt = M_next_list[p]
                        if v_anc == 'omega':
                            continue
                        elif v_nxt == 'omega':
                            estritamente_maior = True
                        elif v_nxt < v_anc:
                            maior_ou_igual = False
                            break
                        elif v_nxt > v_anc:
                            estritamente_maior = True
                            
                    if maior_ou_igual and estritamente_maior:
                        # Introduz omega nas coordenadas que cresceram
                        for p in range(len(rede.lugares)):
                            if anc_M[p] != 'omega' and M_next_list[p] != 'omega':
                                if M_next_list[p] > anc_M[p]:
                                    M_next_list[p] = 'omega'
                                    omega_detectado = True
                        break
                        
                M_final = tuple(M_next_list)
                
                # Verificar se nó já existe na árvore para evitar expansão infinita
                if not any(n['M'] == M_final for n in nos_gerados):
                    novo_no = {
                        'id': len(nos_gerados),
                        'M': M_final,
                        'parent_id': curr['id'],
                        'transicao': t_nome,
                        'caminho': curr['caminho'] + [M_final]
                    }
                    nos_gerados.append(novo_no)
                    fila.append(novo_no)
                    
    return nos_gerados, omega_detectado

nos_cobertura, tem_omega = construir_arvore_cobertura(rede_aberta)

print(f"✅ Árvore de Cobertura gerada com {len(nos_cobertura)} nós.")
print(f"⚠️ Símbolo ω (ômega) detectado? {tem_omega}")

# Exibir os primeiros nós da Árvore de Cobertura
df_cobertura = pd.DataFrame([
    {
        'Nó ID': n['id'],
        'Transição Disparada': n['transicao'] if n['transicao'] else 'INÍCIO (M0)',
        'Marcação [p1, p2, p3, p4, p5]': str(n['M'])
    }
    for n in nos_cobertura[:8]
])
display(df_cobertura)

# Asserção: comprovar que o símbolo omega surgiu estritamente na coordenada da canaleta
assert tem_omega == True, "Erro: O algoritmo deveria ter detectado o crescimento ilimitado (omega)!"
assert any(n['M'][4] == 'omega' for n in nos_cobertura), "Falha: O símbolo omega deve residir no lugar p5 (Canaleta)!"
print("\n✅ Asserção validada: Comprovado risco de transbordo físico (omega no lugar p5).")

---
## 3. Parte B: Rede Controlada com Lugar Complementar ($p_6$) e Árvore de Atingibilidade Finita

Adicionamos o lugar **$p_6$ (`VAGAS_CANALETA`)** com marcação inicial $M_0(p_6) = 3$, modelando a capacidade máxima física da canaleta com o sensor `FULL-300`. A transição $t_2$ (desvio pneumático) agora requer obrigatoriamente $1$ vaga livre em $p_6$.

In [ ]:
lugares_controlada = ["p1_SILO", "p2_ESTEIRA", "p3_TUNEL", "p4_PISTAO", "p5_CANALETA", "p6_VAGAS"]
transicoes_controlada = ["t1_ALIMENTAR", "t2_DESVIAR", "t3_COLETAR", "t4_REABASTECER"]

# Matriz Pre com lugar complementar p6
pre_controlada = [
    [1, 0, 0, 0],  # p1
    [1, 0, 0, 0],  # p2
    [0, 1, 0, 0],  # p3
    [0, 1, 0, 0],  # p4
    [0, 0, 1, 0],  # p5
    [0, 1, 0, 0],  # p6: consome 1 vaga ao desviar!
]

# Matriz Post com restituição de vaga em t3
post_controlada = [
    [0, 0, 0, 1],  # p1
    [0, 1, 0, 0],  # p2
    [1, 0, 0, 0],  # p3
    [0, 1, 0, 0],  # p4
    [0, 1, 0, 0],  # p5
    [0, 0, 1, 0],  # p6: restitui 1 vaga quando o AGV/operador recolhe peça!
]

m0_controlada = (1, 1, 0, 1, 0, 3)  # Canaleta inicia com 3 vagas livres
rede_controlada = RedePetriFMS(lugares_controlada, transicoes_controlada, pre_controlada, post_controlada, m0_controlada)

df_incidencia_controlada = pd.DataFrame(rede_controlada.C, index=lugares_controlada, columns=transicoes_controlada)
print("=== MATRIZ DE INCIDÊNCIA C (REDE CONTROLADA - COM SENSOR FULL-300) ===")
display(df_incidencia_controlada)

### Construção da Árvore de Atingibilidade Completa

In [ ]:
def construir_arvore_atingibilidade(rede: RedePetriFMS) -> Tuple[List[Tuple[int, ...]], List[dict], nx.DiGraph]:
    """
    Gera o conjunto completo e finito de marcações atingíveis R(N, M0)
    e constrói o Grafo de Atingibilidade correspondente.
    """
    marcacoes_conhecidas = [rede.m0]
    fila = deque([rede.m0])
    arestas_grafo = []
    G_reach = nx.DiGraph()
    G_reach.add_node(rede.m0)
    
    while fila:
        M_curr = fila.popleft()
        
        for t_idx, t_nome in enumerate(rede.transicoes):
            if rede.habilitada(M_curr, t_idx):
                M_next = rede.disparar(M_curr, t_idx)
                arestas_grafo.append({
                    'origem': M_curr,
                    'transicao': t_nome,
                    'destino': M_next
                })
                G_reach.add_edge(M_curr, M_next, label=t_nome)
                
                if M_next not in marcacoes_conhecidas:
                    marcacoes_conhecidas.append(M_next)
                    fila.append(M_next)
                    
    return marcacoes_conhecidas, arestas_grafo, G_reach

marcacoes_atingiveis, transicoes_grafo, G_reach = construir_arvore_atingibilidade(rede_controlada)

print(f"✅ Árvore de Atingibilidade gerada com sucesso!")
print(f"📊 Total de Marcações Físicas Atingíveis |R(N, M0)| = {len(marcacoes_atingiveis)} estados distintos.")
print(f"🔗 Total de Transições Ativas no Grafo = {len(transicoes_grafo)} arcos direcionados.")

---
## 4. Auditoria Formal de Propriedades com Asserções (`assert`)

Comprovamos formalmente:
1. **Invariante de Lugar (P-Invariant):** $M(p_5) + M(p_6) = 3$ em $100\%$ dos estados.
2. **Limitação Estrita:** A canaleta nunca ultrapassa $3$ peças ($M(p_5) \le 3$).
3. **Segurança (*Safeness*):** Os lugares da esteira $p_2, p_3, p_4$ são estritamente $1$-limitados ($M(p_i) \in \{0, 1\}$).
4. **Ausência de *Deadlocks*:** Todo nó possui pelo menos uma transição de saída viável.

In [ ]:
# Asserção 1: Comprovação do Invariante de Lugar da Canaleta (P-Invariant)
for M in marcacoes_atingiveis:
    soma_canaleta = M[4] + M[5]
    assert soma_canaleta == 3, f"Violação do Invariante-P na marcação {M}: p5 + p6 = {soma_canaleta} != 3"

# Asserção 2: Limitação Física e Ausência de Transbordo
max_pecas_canaleta = max(M[4] for M in marcacoes_atingiveis)
assert max_pecas_canaleta == 3, f"Falha de capacidade: Canaleta excedeu 3 peças! Máximo: {max_pecas_canaleta}"

# Asserção 3: 1-Limitação dos Atuadores e Esteira (Safeness)
for M in marcacoes_atingiveis:
    assert M[1] <= 1, f"Colisão mecânica: Mais de uma peça na esteira no estado {M}!"
    assert M[2] <= 1, f"Sobrecarga no túnel óptico: Mais de uma peça inspecionada no estado {M}!"
    assert M[3] <= 1, f"Conflito pneumático: Pistão em estado incoerente no estado {M}!"

# Asserção 4: Ausência de Deadlocks (Travamentos de Processo)
for M in marcacoes_atingiveis:
    tem_transicao_ativa = any(rede_controlada.habilitada(M, t) for t in range(len(rede_controlada.transicoes)))
    assert tem_transicao_ativa == True, f"DEADLOCK CRÍTICO DETECTADO: Estado {M} não possui transições habilitadas!"

# Asserção 5: Identificar o estado de Canaleta Cheia e verificar que t2 (desvio) fica BLOQUEADA
estados_cheios = [M for M in marcacoes_atingiveis if M[4] == 3]
assert len(estados_cheios) > 0, "Deveria haver pelo menos um estado com canaleta cheia"
for M_cheio in estados_cheios:
    assert rede_controlada.habilitada(M_cheio, 1) == False, f"Falha de Segurança: t2 (Desvio) habilitada com canaleta cheia em {M_cheio}!"

print("✅ Todas as 5 asserções matemáticas foram comprovadas com 100% de sucesso!")
print("🏆 Conclusão Formal: O sistema é Bounded, Safe, Deadlock-Free e protegido contra transbordamento.")

---
## 5. Visualização do Grafo de Atingibilidade da Esteira Seletora

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))

# Layout circular para destacar a ciclicidade reversível da esteira
pos_reach = nx.spring_layout(G_reach, seed=42, k=1.8)

# Cores dos nós baseadas no status da canaleta
cores_nos = []
for M in G_reach.nodes():
    if M == rede_controlada.m0:
        cores_nos.append('#38bdf8')  # Raiz / Início (Azul Ciano)
    elif M[4] == 3:
        cores_nos.append('#ef4444')  # Canaleta Cheia (Vermelho Alerta)
    else:
        cores_nos.append('#10b981')  # Operação Normal (Verde)

# Desenhar arestas direcionadas
nx.draw_networkx_edges(G_reach, pos_reach, edge_color='#64748b', arrowsize=15, width=1.5, ax=ax)

# Desenhar nós
nx.draw_networkx_nodes(G_reach, pos_reach, node_color=cores_nos, node_size=1800, edgecolors='black', linewidths=1.5, ax=ax)

# Rótulos das marcações resumidas
rotulos = {}
for M in G_reach.nodes():
    rotulos[M] = f"Silo:{M[0]}|Est:{M[1]}\nTun:{M[2]}|Pis:{M[3]}\nCX:{M[4]}|Vag:{M[5]}"
nx.draw_networkx_labels(G_reach, pos_reach, labels=rotulos, font_size=7, font_color='black', font_weight='bold', ax=ax)

# Rótulos das transições nos arcos
edge_labels = nx.get_edge_attributes(G_reach, 'label')
nx.draw_networkx_edge_labels(G_reach, pos_reach, edge_labels=edge_labels, font_size=7, font_color='#0f172a', ax=ax)

ax.set_title("Grafo de Atingibilidade da Esteira Seletora de Peças (Rede de Petri Controlada)", fontsize=13, fontweight='bold')

# Legenda explicativa
from matplotlib.lines import Line2D
legenda_elementos = [
    Line2D([0], [0], marker='o', color='w', label='M0 (Marcação Inicial)', markerfacecolor='#38bdf8', markersize=12),
    Line2D([0], [0], marker='o', color='w', label='Marcações Nominais de Operação', markerfacecolor='#10b981', markersize=12),
    Line2D([0], [0], marker='o', color='w', label='Canaleta Cheia (p5=3, p6=0 - Desvio Inibido)', markerfacecolor='#ef4444', markersize=12)
]
ax.legend(handles=legenda_elementos, loc='upper left', framealpha=0.95, fontsize=9)
ax.axis('off')

plt.tight_layout()
plt.show()

---
## 6. Conclusões de Engenharia de Automação

1. **Prevenção de Transbordo Comprovada:** A análise da Árvore de Cobertura demonstrou que a rede aberta gera $\omega$ no lugar da canaleta. A introdução do sensor `FULL-300` como lugar complementar $p_6$ fixou o Invariante de Lugar $M(p_5) + M(p_6) = 3$, delimitando fisicamente o loteamento em $3$ peças.
2. **Imunidade a Travamentos (*Deadlock-Free*):** A Árvore de Atingibilidade finita provou matematicamente que mesmo no estado de canaleta cheia, a transição de coleta $t_3$ permanece viva, permitindo ao AGV esvaziar a baia e restaurar o fluxo livre de produção sem intervenções de emergência.